# <font color="#418FDE" size="6.5" uppercase>**Naive Bayes**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Choose an appropriate Naive Bayes variant for numeric, count, binary, or categorical features. 
- Configure smoothing, priors, and sample weights in Naive Bayes models. 
- Use partial_fit for incremental Naive Bayes training. 


## **1. Naive Bayes Assumptions**

### **1.1. Conditional Independence**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_01_01.jpg?v=1784005172" width="250">



>* Features are independent once class is known
>* Simplifies classification despite imperfect real-world assumptions

>* Independence is assumed within each class
>* Individual feature evidence scales to many features

>* Match Naive Bayes variants to feature types.
>* Use independence as a practical approximation.



### **1.2. Gaussian Feature Modeling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_01_02.jpg?v=1784005174" width="250">



>* Models continuous features with class-specific bell curves
>* Converts distance from averages into likelihoods

>* Best for continuous, roughly bell-shaped measurements
>* Combines class-specific feature profiles for prediction

>* Skewed or outlier-heavy data can mislead
>* Use Gaussian only for true continuous measurements



In [ ]:
#@title Python Code - Gaussian Feature Modeling

# This example models continuous measurements with Gaussian Naive Bayes.
# It compares class-specific centers and spreads visually.
# The result shows likelihood-shaped feature profiles.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import sklearn

# Load a small dataset of continuous flower measurements.
iris = load_iris()
feature_index = 2

# Keep one numeric feature for a focused Gaussian example.
X = iris.data[:, [feature_index]]
y = iris.target

# Validate the simple shape expected by this lesson.
if X.shape[1] != 1 or len(y) != len(X):
    raise ValueError("Expected one feature and one label per row.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=42
)

# Fit Gaussian Naive Bayes to continuous numeric measurements.
model = GaussianNB()
model.fit(X_train, y_train)

# Evaluate the model with one concise accuracy result.
predicted = model.predict(X_test)
accuracy = accuracy_score(y_test, predicted)

print("scikit-learn version:", sklearn.__version__)
print("Feature modeled: petal length in centimeters")
print("Test accuracy:", round(accuracy, 3))

# Build smooth Gaussian curves from learned means and variances.
x_values = np.linspace(X.min() - 0.2, X.max() + 0.2, 300)
fig, ax = plt.subplots(figsize=(8, 5))

# Plot each class profile learned by Gaussian Naive Bayes.
for class_index, class_name in enumerate(iris.target_names):
    mean = model.theta_[class_index, 0]
    variance = model.var_[class_index, 0]
    density = np.exp(-0.5 * ((x_values - mean) ** 2) / variance)
    density = density / np.sqrt(2 * np.pi * variance)
    ax.plot(x_values, density, label=class_name)

# Label the single plot for beginner interpretation.
ax.set_title("Gaussian Naive Bayes feature modeling")
ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Estimated likelihood density")
ax.legend()

plt.show()



### **1.3. Feature Likelihoods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_01_03.jpg?v=1784005176" width="250">



>* Likelihoods link features to possible classes
>* Match Naive Bayes variant to feature type

>* Match likelihoods to feature data types
>* Wrong representations create misleading class evidence

>* Likelihoods shape each feature’s class evidence
>* Match variants to feature representation



## **2. Naive Bayes Variants**

### **2.1. Multinomial Smoothing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_02_01.jpg?v=1784005166" width="250">



>* Count features reveal class associations
>* Smoothing prevents zero-probability overconfidence

>* Add small counts to avoid zero probabilities
>* Tune smoothing to balance rare evidence

>* Match smoothing to count data quality
>* Use priors and weights for calibration



In [ ]:
#@title Python Code - Multinomial Smoothing

# This example shows smoothing in Multinomial Naive Bayes.
# Smoothing protects predictions from unseen word counts.
# Different alpha values produce different class probabilities.

import numpy as np
import sklearn
from sklearn.naive_bayes import MultinomialNB

# Rows are tiny documents, and columns are word counts.
feature_names = np.array(["sale", "prize", "meeting", "project"])
X_train = np.array([[3, 2, 0, 0], [2, 1, 0, 0], [0, 0, 2, 3], [0, 0, 3, 2]])
y_train = np.array(["spam", "spam", "work", "work"])

# This new document contains an unseen spam word, meeting.
X_new = np.array([[0, 0, 1, 0]])
new_word = feature_names[np.argmax(X_new[0])]

# Validate the small count matrix before fitting models.
if X_train.shape != (4, 4) or X_new.shape != (1, 4):
    raise ValueError("Expected four training rows and four count features.")

# Compare no smoothing with common Laplace smoothing.
model_no_smoothing = MultinomialNB(alpha=0.0, force_alpha=True)
model_laplace = MultinomialNB(alpha=1.0)

model_no_smoothing.fit(X_train, y_train)
model_laplace.fit(X_train, y_train)

# Predict probabilities for the same document under both settings.
classes = model_laplace.classes_
probs_no_smoothing = model_no_smoothing.predict_proba(X_new)[0]
probs_laplace = model_laplace.predict_proba(X_new)[0]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"New document word: {new_word}")
print(f"Classes: {classes[0]}, {classes[1]}")
print(f"alpha=0.0 probabilities: {np.round(probs_no_smoothing, 3)}")
print(f"alpha=1.0 probabilities: {np.round(probs_laplace, 3)}")
print("Smoothing keeps an unseen word from forcing probability to zero.")



### **2.2. Complement NB**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_02_02.jpg?v=1784005170" width="250">



>* Improves classification when classes are imbalanced
>* Uses other classes to highlight differences

>* Smoothing prevents extreme rare-feature weights
>* Tune smoothing for data size and noise

>* Set priors to reflect class prevalence
>* Weight samples by reliability or importance



In [ ]:
#@title Python Code - Complement NB

# This example compares Complement Naive Bayes settings.
# Smoothing and sample weights change learned probabilities.
# The output shows accuracy and rare-class recall.

import numpy as np
import sklearn
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB

# Rows are short count-based messages with three word groups.
X = np.array([
    [8, 1, 0], [7, 2, 0], [6, 1, 0], [7, 1, 1],
    [1, 7, 0], [0, 8, 1], [1, 6, 0], [0, 7, 1],
    [1, 0, 6], [0, 1, 7], [1, 1, 8], [0, 0, 7],
    [2, 0, 5], [1, 0, 4], [0, 2, 5], [1, 1, 5],
    [6, 0, 1], [5, 1, 1], [0, 6, 1], [1, 5, 1],
])

# Class two is intentionally rare in this tiny dataset.
y = np.array([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 1, 1])

# The split keeps each class represented in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=42
)

# Validate the small count matrix before fitting.
if X_train.shape[1] != 3 or np.any(X_train < 0):
    raise ValueError("Expected three nonnegative count features.")

# Give rare-class training examples extra influence.
sample_weight = np.ones(len(y_train))
sample_weight[y_train == 2] = 3.0

# Compare default smoothing with weighted rare-class evidence.
default_model = ComplementNB(alpha=1.0)
weighted_model = ComplementNB(alpha=0.5)

default_model.fit(X_train, y_train)
weighted_model.fit(X_train, y_train, sample_weight=sample_weight)

# Evaluate overall accuracy and recall for the rare class.
default_pred = default_model.predict(X_test)
weighted_pred = weighted_model.predict(X_test)

print("scikit-learn version:", sklearn.__version__)
print("Default accuracy:", round(accuracy_score(y_test, default_pred), 2))
print("Weighted accuracy:", round(accuracy_score(y_test, weighted_pred), 2))
print("Default rare-class recall:", round(recall_score(y_test, default_pred, labels=[2], average="macro"), 2))
print("Weighted rare-class recall:", round(recall_score(y_test, weighted_pred, labels=[2], average="macro"), 2))



### **2.3. Bernoulli Categorical**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_02_03.jpg?v=1784005168" width="250">



>* Models yes-or-no feature presence
>* Smoothing prevents impossible unseen combinations

>* Categorical NB models fixed category values
>* Smoothing balances rare or unseen categories

>* Priors align class expectations with reality
>* Sample weights control each example’s influence



In [ ]:
#@title Python Code - Bernoulli Categorical

# Compare smoothing, priors, and sample weights.
# Bernoulli features record yes or no evidence.
# Categorical features record one named choice.

import numpy as np
import sklearn
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

# Small binary data marks whether each signal is present.
X_binary = np.array([
    [1, 1, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1],
    [0, 0, 1], [0, 1, 1], [1, 0, 1], [0, 0, 0]
])

# Labels use zero for normal and one for alert.
y = np.array([1, 1, 1, 0, 0, 0, 0, 0])

# Recent or trusted examples can count more.
sample_weight = np.array([2, 2, 1, 1, 1, 1, 1, 1])

# The test case has two alert-like signals present.
test_binary = np.array([[1, 1, 0]])

# Smoothing prevents unseen combinations from becoming impossible.
plain_model = BernoulliNB(alpha=1.0)
plain_model.fit(X_binary, y)

weighted_model = BernoulliNB(alpha=1.0, class_prior=[0.8, 0.2])
weighted_model.fit(X_binary, y, sample_weight=sample_weight)

plain_alert = plain_model.predict_proba(test_binary)[0, 1]
weighted_alert = weighted_model.predict_proba(test_binary)[0, 1]

# Categorical data stores one category code per feature.
X_category = np.array([
    [0, 0], [0, 1], [1, 0], [2, 2],
    [2, 1], [1, 2], [0, 2], [2, 0]
])

# The categorical test case uses fixed category values.
test_category = np.array([[0, 0]])

category_model = CategoricalNB(alpha=1.0, class_prior=[0.8, 0.2])
category_model.fit(X_category, y, sample_weight=sample_weight)

category_alert = category_model.predict_proba(test_category)[0, 1]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Bernoulli alert probability, learned priors: {plain_alert:.3f}")
print(f"Bernoulli alert probability, prior plus weights: {weighted_alert:.3f}")
print(f"Categorical alert probability, prior plus weights: {category_alert:.3f}")
print("Smoothing alpha=1.0 keeps rare feature values from forcing zero probability.")



## **3. Incremental NB Training**

### **3.1. Smoothing for Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_03_01.jpg?v=1784005162" width="250">



>* Early batches may miss rare feature values
>* Smoothing prevents zero-probability overconfidence

>* Smoothing stabilizes early incremental updates
>* Its influence fades as evidence grows

>* Match smoothing to batches and features
>* Keep smoothing consistent across incremental updates



In [ ]:
#@title Python Code - Smoothing for Updates

# Demonstrate smoothing during incremental Naive Bayes updates.
# Compare zero smoothing with stable additive smoothing.
# Show probabilities after each small training batch.

import numpy as np
import sklearn
from sklearn.naive_bayes import MultinomialNB

# Each row stores counts for two simple word features.
X_batches = [
    np.array([[3, 0], [2, 0], [0, 2], [0, 3]]),
    np.array([[1, 1], [0, 4], [4, 1], [3, 2]]),
]

y_batches = [
    np.array([0, 0, 1, 1]),
    np.array([0, 1, 0, 1]),
]

# The classes must be supplied on the first partial_fit call.
classes = np.array([0, 1])
models = {
    "alpha=0.0": MultinomialNB(alpha=0.0, force_alpha=True),
    "alpha=1.0": MultinomialNB(alpha=1.0),
}

# Validate that every batch has the same two feature columns.
for batch in X_batches:
    if batch.shape[1] != 2:
        raise ValueError("Each batch must contain exactly two features.")

print("scikit-learn version:", sklearn.__version__)
print("Probability of feature 2 given class 0 after each update:")

# Update each model one batch at a time.
for name, model in models.items():
    shown_values = []
    for batch_number in range(len(X_batches)):
        model.partial_fit(
            X_batches[batch_number], y_batches[batch_number], classes=classes
        )
        probability = np.exp(model.feature_log_prob_[0, 1])
        shown_values.append(round(float(probability), 3))
    print(name, shown_values)

# The smoothed model avoids an early zero probability.
print("Smoothing keeps early missing features possible during updates.")



### **3.2. Priors and Weights**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_03_02.jpg?v=1784005164" width="250">



>* Priors set initial class expectations
>* Manage priors to reduce batch-order bias

>* Weight examples by importance or reliability
>* Reflect priorities without full retraining

>* Balance priors with sample weights deliberately
>* Match updates to real class patterns



In [ ]:
#@title Python Code - Priors and Weights

# Demonstrate incremental Naive Bayes updates.
# Compare priors and sample weights.
# Show how early batches influence predictions.

import numpy as np
import sklearn
from sklearn.naive_bayes import GaussianNB

# This tiny stream has two numeric features and two classes.
X_batch_one = np.array([[0.8, 1.0], [1.0, 0.9], [1.2, 1.1], [1.1, 0.8]])
y_batch_one = np.array([0, 0, 0, 0])

X_batch_two = np.array([[3.0, 3.2], [3.2, 2.9], [2.8, 3.1], [3.1, 3.0]])
y_batch_two = np.array([1, 1, 1, 1])

# This point is closer to class one examples.
test_point = np.array([[2.7, 2.9]])
classes = np.array([0, 1])

# A fixed prior says both classes are equally likely initially.
plain_model = GaussianNB(priors=[0.5, 0.5])
weighted_model = GaussianNB(priors=[0.5, 0.5])

# The first update sees only class zero examples.
plain_model.partial_fit(X_batch_one, y_batch_one, classes=classes)
weighted_model.partial_fit(X_batch_one, y_batch_one, classes=classes)

# The second update gives rare class one examples extra influence.
class_one_weights = np.array([3.0, 3.0, 3.0, 3.0])
plain_model.partial_fit(X_batch_two, y_batch_two)
weighted_model.partial_fit(
    X_batch_two,
    y_batch_two,
    sample_weight=class_one_weights,
)

plain_probability = plain_model.predict_proba(test_point)[0, 1]
weighted_probability = weighted_model.predict_proba(test_point)[0, 1]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Plain incremental P(class 1): {plain_probability:.3f}")
print(f"Weighted incremental P(class 1): {weighted_probability:.3f}")
print("Sample weights made the later rare class update count more.")



### **3.3. Incremental Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_03_03.jpg?v=1784005160" width="250">



>* Update models as new data arrives
>* Naive Bayes efficiently accumulates class statistics

>* Declare all classes before incremental training
>* Keep feature columns consistent across batches

>* Handle drift with recent weighted updates
>* Monitor validation performance to avoid reinforcing noise



In [ ]:
#@title Python Code - Incremental Updates

# Demonstrate incremental Naive Bayes updates.
# Use batches with partial_fit training.
# Compare learning after each batch.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

# Create a small numeric classification dataset.
features, labels = make_classification(
    n_samples=240,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    n_classes=3,
    random_state=42,
)

# Split once so testing stays separate.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42,
)

# Confirm the class list is complete.
all_classes = np.array([0, 1, 2])
if len(np.unique(train_labels)) != len(all_classes):
    raise ValueError("Training data must contain all expected classes.")

# Train the same model in three batches.
model = GaussianNB()
batch_size = 60
accuracies = []

for batch_number in range(3):
    start = batch_number * batch_size
    stop = start + batch_size
    batch_features = train_features[start:stop]
    batch_labels = train_labels[start:stop]

    # The first partial_fit call must receive all classes.
    if batch_number == 0:
        model.partial_fit(batch_features, batch_labels, classes=all_classes)
    else:
        model.partial_fit(batch_features, batch_labels)

    predictions = model.predict(test_features)
    accuracy = accuracy_score(test_labels, predictions)
    accuracies.append(round(accuracy, 3))

print("scikit-learn version:", sklearn.__version__)
print("Test accuracy after each partial_fit batch:", accuracies)
print("Total training rows seen incrementally:", batch_size * len(accuracies))



# <font color="#418FDE" size="6.5" uppercase>**Naive Bayes**</font>


In this lecture, you learned to:
- Choose an appropriate Naive Bayes variant for numeric, count, binary, or categorical features. 
- Configure smoothing, priors, and sample weights in Naive Bayes models. 
- Use partial_fit for incremental Naive Bayes training. 

In the next Lecture (Lecture B), we will go over 'Probability Calibration'